**Sample ID**: 150

**Query**:

I need to make some changes to my orders. I want to exchange the bicycle and jigsaw puzzle from the same order, exchange my camera, and cancel my skateboard order.

**DB Type**: Base Case

**Case Description**:

The user, Sofia Li (zip code 78260), is requesting modifications for three separate orders. For order #W3916020, she wants to exchange two items: a bicycle ('7758198585') for one with a larger frame size ('5606522780') and a jigsaw puzzle ('4068787148') for one with the same difficulty but 1000 more pieces, preferring an animal theme over art ('6245746168'). For order #W4689314, she is exchanging a camera ('5996159312') for a model with a slightly lower resolution ('8363011723'). Finally, she is canceling the pending skateboard order #W8855135, stating it is no longer needed. If a single item cancellation is not possible for the skateboard order, she is okay with canceling the entire order.





```
<multiturn info>
User Details: Sofia Li, zip code 78260 (Information Gathering)
Exchange Request 1: Bicycle and Jigsaw Puzzle from the same order. (Information Gathering)
Bicycle Exchange Criteria: Exchange for a larger frame size for my kid. (Information Gathering)
Puzzle Exchange Criteria: Exchange for 1000 more pieces, same difficulty, and prefer an animal over an art theme if both are available. (Information Gathering)
Exchange Request 2: Camera. (Information Gathering)
Camera Exchange Criteria: Exchange for a slightly lower resolution, without changing the other options. (Information Gathering)
Cancellation Request: Skateboard order. (Information Gathering)
Cancellation Contingency: If the skateboard cannot be cancelled individually, cancel the entire order with the reason 'no longer needed'. (Information Gathering)
Payment Clarification: If asked for (Confirmation), will request to use the other card for payment/refund. (Follow Up Request)
</multiturn info>
```

**Global/Context Variables:**


**APIs:**

- retail


# Set Up

## Download relevant files

In [ ]:
import io
import os
import sys
import zipfile
import shutil
import re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Version to download
VERSION = "0.1.1"  # Version of the API

# Define paths
CONTENT_DIR = '/content'
APIS_DIR = os.path.join(CONTENT_DIR, 'APIs')
DBS_DIR = os.path.join(CONTENT_DIR, 'DBs')
SCRIPTS_DIR = os.path.join(CONTENT_DIR, 'Scripts')
FC_DIR = os.path.join(CONTENT_DIR, 'Schemas')
ZIP_PATH = os.path.join(CONTENT_DIR, f'APIs_V{VERSION}.zip')

# Google Drive Folder ID where versioned APIs zip files are stored
APIS_FOLDER_ID = '1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4'

# List of items to extract from the zip file
ITEMS_TO_EXTRACT = ['APIs/', 'DBs/', 'Scripts/', 'Schemas/']

# Clean up existing directories and files
for path in [APIS_DIR, DBS_DIR, SCRIPTS_DIR, FC_DIR, ZIP_PATH]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

# Authenticate and create the drive service
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Helper function to download a file from Google Drive
def download_drive_file(service, file_id, output_path, file_name=None, show_progress=True):
    """Downloads a file from Google Drive"""
    destination = output_path
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(destination, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if show_progress:
                print(f"Download progress: {int(status.progress() * 100)}%")


# 1. List files in the specified APIs folder
print(f"Searching for APIs zip file with version {VERSION} in folder: {APIS_FOLDER_ID}...")
apis_file_id = None

try:
    query = f"'{APIS_FOLDER_ID}' in parents and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    for file in files:
        file_name = file.get('name', '')
        if file_name.lower() == f'apis_v{VERSION.lower()}.zip':
            apis_file_id = file.get('id')
            print(f"Found matching file: {file_name} (ID: {apis_file_id})")
            break

except Exception as e:
    print(f"An error occurred while listing files in Google Drive: {e}")

if not apis_file_id:
    print(f"Error: Could not find APIs zip file with version {VERSION} in the specified folder.")
    sys.exit("Required APIs zip file not found.")

# 2. Download the found APIs zip file
print(f"Downloading APIs zip file with ID: {apis_file_id}...")
download_drive_file(drive_service, apis_file_id, ZIP_PATH, file_name=f'APIs_V{VERSION}.zip')

# 3. Extract specific items from the zip file to /content
print(f"Extracting specific items from {ZIP_PATH} to {CONTENT_DIR}...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()

        for member in zip_contents:
            extracted = False
            for item_prefix in ITEMS_TO_EXTRACT:
              if member == item_prefix or member.startswith(item_prefix):
                    zip_ref.extract(member, CONTENT_DIR)
                    extracted = True
                    break

except zipfile.BadZipFile:
    print(f"Error: The downloaded file at {ZIP_PATH} is not a valid zip file.")
    sys.exit("Invalid zip file downloaded.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")
    sys.exit("Extraction failed.")


# 4. Clean up
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# 5. Add APIs to path
if os.path.exists(APIS_DIR):
    sys.path.append(APIS_DIR)
else:
    print(f"Error: APIS directory not found at {APIS_DIR} after extraction. Cannot add to path.")

# 6. Quick verification
# Check for the presence of the extracted items
verification_paths = [APIS_DIR, DBS_DIR, SCRIPTS_DIR]
all_present = True
print("\nVerifying extracted items:")
for path in verification_paths:
    if os.path.exists(path):
        print(f"✅ {path} is present.")
    else:
        print(f"❌ {path} is MISSING!")
        all_present = False

if all_present:
    print(f"\n✅ Setup complete! Required items extracted to {CONTENT_DIR}.")
else:
    print("\n❌ Setup failed! Not all required items were extracted.")

# 7. Generate Schemas

print("\nGenerating FC Schemas")

# Change working directory to the source folder

# Iterate through the packages in the /content/APIs directory

    # Check if it's a directory (to avoid processing files)
        # Call the function to generate schema for the current package
print(f"✅ Successfully generated {len(os.listdir(FC_DIR))} FC Schemas to {FC_DIR}")
os.chdir(CONTENT_DIR)

Searching for APIs zip file with version 0.1.1 in folder: 1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4...
Found matching file: APIs_V0.1.1.zip (ID: 1qDwJfS_gCyw6Z3HA1llw2KYZMwNHNHsi)
Download progress: 100%
Extracting specific items from /content/APIs_V0.1.1.zip to /content...

Verifying extracted items:
✅ /content/APIs is present.
✅ /content/DBs is present.
✅ /content/Scripts is present.

✅ Setup complete! Required items extracted to /content.

Generating FC Schemas
✅ Successfully generated 68 FC Schemas to /content/Schemas


## Install Dependencies and Clone Repositories

In [ ]:
!pip install -r /content/APIs/requirements.txt

## Import APIs and initiate DBs

In [ ]:
# Using default DB for the Tau Benchmark
import retail

retail.SimulationEngine.db.load_state("/content/DBs/RetailDefaultDB.json")


# Initial Assertion

1. Order #W3916020 is in a 'delivered' state and contains a bicycle with item ID '7758198585' and a jigsaw puzzle with item ID '4068787148'.
2. The replacement bicycle with item ID '5606522780' and the replacement jigsaw puzzle with item ID '6245746168' are available.
3. Order #W4689314 is in a 'delivered' state and contains a camera with item ID '5996159312'.
4. The replacement camera with item ID '8363011723' is available.
5. Order #W8855135 is in a 'pending' state and contains a skateboard.

In [ ]:
import retail
from Scripts.assertions_utils import *

# Define constants for order and item identifiers
ORDER_ID_1 = '#W3916020'
ITEM_ID_BICYCLE_ORIGINAL = '7758198585'
ITEM_ID_PUZZLE_ORIGINAL = '4068787148'
ITEM_ID_BICYCLE_REPLACEMENT = '5606522780'
ITEM_ID_PUZZLE_REPLACEMENT = '6245746168'

ORDER_ID_2 = '#W4689314'
ITEM_ID_CAMERA_ORIGINAL = '5996159312'
ITEM_ID_CAMERA_REPLACEMENT = '8363011723'

ORDER_ID_3 = '#W8855135'
ITEM_NAME_SKATEBOARD = 'Skateboard'

DELIVERED_STATUS = 'delivered'
PENDING_STATUS = 'pending'

# --- Data Gathering ---
# Initialize variables to store data and potential errors
order1_details = None
order2_details = None
order3_details = None
replacement_bicycle_product_details = None
replacement_puzzle_product_details = None
replacement_camera_product_details = None
api_call_error = None

try:
    # Fetch details for the first order (#W3916020)
    order1_details = retail.get_order_details(order_id=ORDER_ID_1)
    if order1_details:
        order1_items = order1_details.get('items', [])
        bicycle_product_id = next((item.get('product_id') for item in order1_items if item.get('item_id') == ITEM_ID_BICYCLE_ORIGINAL), None)
        puzzle_product_id = next((item.get('product_id') for item in order1_items if item.get('item_id') == ITEM_ID_PUZZLE_ORIGINAL), None)

        if bicycle_product_id:
            replacement_bicycle_product_details = retail.get_product_details(product_id=bicycle_product_id)
        if puzzle_product_id:
            replacement_puzzle_product_details = retail.get_product_details(product_id=puzzle_product_id)

    # Fetch details for the second order (#W4689314)
    order2_details = retail.get_order_details(order_id=ORDER_ID_2)
    if order2_details:
        order2_items = order2_details.get('items', [])
        camera_product_id = next((item.get('product_id') for item in order2_items if item.get('item_id') == ITEM_ID_CAMERA_ORIGINAL), None)

        if camera_product_id:
            replacement_camera_product_details = retail.get_product_details(product_id=camera_product_id)

    # Fetch details for the third order (#W8855135)
    order3_details = retail.get_order_details(order_id=ORDER_ID_3)

except Exception as e:
    api_call_error = str(e)

# --- Assertions ---

# --- Assertion 1: Order #W3916020 is in a 'delivered' state and contains a bicycle with item ID '7758198585' and a jigsaw puzzle with item ID '4068787148'. ---
order1_status_is_pending = False
order1_has_bicycle = False
order1_has_puzzle = False
if order1_details:
    order1_status_is_pending = compare_strings(order1_details.get('status'),DELIVERED_STATUS)
    order1_item_ids = [item.get('item_id') for item in order1_details.get('items', [])]
    order1_has_bicycle = compare_is_list_subset(ITEM_ID_BICYCLE_ORIGINAL, order1_item_ids,'any')
    order1_has_puzzle = compare_is_list_subset(ITEM_ID_PUZZLE_ORIGINAL,order1_item_ids,'any')

assertion_condition_1 = order1_status_is_pending and order1_has_bicycle and order1_has_puzzle
assertion_message_1 = f"Assertion 1 Failed: Order {ORDER_ID_1} initial state is incorrect. "
if api_call_error:
    assertion_message_1 += f"API call failed: {api_call_error}."
else:
    assertion_message_1 += f"Expected status: '{DELIVERED_STATUS}', Actual: '{order1_details.get('status', 'N/A')}'. "
    assertion_message_1 += f"Expected bicycle '{ITEM_ID_BICYCLE_ORIGINAL}', Found: {order1_has_bicycle}. "
    assertion_message_1 += f"Expected puzzle '{ITEM_ID_PUZZLE_ORIGINAL}', Found: {order1_has_puzzle}."
assert assertion_condition_1, assertion_message_1

# --- Assertion 2: The replacement bicycle with item ID '5606522780' and the replacement jigsaw puzzle with item ID '6245746168' are available. ---
replacement_bicycle_available = False
replacement_puzzle_available = False
if replacement_bicycle_product_details:
    variants = replacement_bicycle_product_details.get('variants', {})
    replacement_bicycle_available = variants.get(ITEM_ID_BICYCLE_REPLACEMENT, {}).get('available', False)
if replacement_puzzle_product_details:
    variants = replacement_puzzle_product_details.get('variants', {})
    replacement_puzzle_available = variants.get(ITEM_ID_PUZZLE_REPLACEMENT, {}).get('available', False)

assertion_condition_2 = replacement_bicycle_available and replacement_puzzle_available
assertion_message_2 = f"Assertion 2 Failed: Replacement items are not available. "
if api_call_error:
    assertion_message_2 += f"API call failed: {api_call_error}."
else:
    assertion_message_2 += f"Bicycle '{ITEM_ID_BICYCLE_REPLACEMENT}' availability: {replacement_bicycle_available}. "
    assertion_message_2 += f"Jigsaw Puzzle '{ITEM_ID_PUZZLE_REPLACEMENT}' availability: {replacement_puzzle_available}."
assert assertion_condition_2, assertion_message_2

# --- Assertion 3: Order #W4689314 is in a 'delivered' state and contains a camera with item ID '5996159312'. ---
order2_status_is_pending = False
order2_has_camera = False
if order2_details:
    order2_status_is_pending = compare_strings(order2_details.get('status'), DELIVERED_STATUS)
    order2_item_ids = [item.get('item_id') for item in order2_details.get('items', [])]
    order2_has_camera = compare_is_list_subset(ITEM_ID_CAMERA_ORIGINAL,order2_item_ids)

assertion_condition_3 = order2_status_is_pending and order2_has_camera
assertion_message_3 = f"Assertion 3 Failed: Order {ORDER_ID_2} initial state is incorrect. "
if api_call_error:
    assertion_message_3 += f"API call failed: {api_call_error}."
else:
    assertion_message_3 += f"Expected status: '{DELIVERED_STATUS}', Actual: '{order2_details.get('status', 'N/A')}'. "
    assertion_message_3 += f"Expected camera '{ITEM_ID_CAMERA_ORIGINAL}', Found: {order2_has_camera}."
assert assertion_condition_3, assertion_message_3

# --- Assertion 4: The replacement camera with item ID '8363011723' is available. ---
replacement_camera_available = False
if replacement_camera_product_details:
    variants = replacement_camera_product_details.get('variants', {})
    replacement_camera_available = variants.get(ITEM_ID_CAMERA_REPLACEMENT, {}).get('available', False)

assertion_condition_4 = replacement_camera_available
assertion_message_4 = f"Assertion 4 Failed: Replacement camera '{ITEM_ID_CAMERA_REPLACEMENT}' is not available. "
if api_call_error:
    assertion_message_4 += f"API call failed: {api_call_error}."
else:
    assertion_message_4 += f"Availability: {replacement_camera_available}."
assert assertion_condition_4, assertion_message_4

# --- Assertion 5: Order #W8855135 is in a 'pending' state and contains a skateboard. ---
order3_status_is_pending = False
order3_has_skateboard = False
if order3_details:
    order3_status_is_pending = compare_strings(order3_details.get('status'),PENDING_STATUS)
    order3_item_names = [item.get('name') for item in order3_details.get('items', [])]
    order3_has_skateboard = compare_is_list_subset(ITEM_NAME_SKATEBOARD,order3_item_names,'any')

assertion_condition_5 = order3_status_is_pending and order3_has_skateboard
assertion_message_5 = f"Assertion 5 Failed: Order {ORDER_ID_3} initial state is incorrect. "
if api_call_error:
    assertion_message_5 += f"API call failed: {api_call_error}."
else:
    assertion_message_5 += f"Expected status: '{PENDING_STATUS}', Actual: '{order3_details.get('status', 'N/A')}'. "
    assertion_message_5 += f"Expected to find item '{ITEM_NAME_SKATEBOARD}', Found: {order3_has_skateboard}."
assert assertion_condition_5, assertion_message_5

# Action

**Simulated User**: I need to make some changes to my orders. I want to exchange the bicycle and jigsaw puzzle in order #W3916020, exchange the camera in order #W4689314, and cancel the skateboard in order #W8855135.

**Action Agent**: To get started, could you please provide me with your email address and zipcode?

 **Simulated User**: I don't have an email address for you, but my name is Sofia Li and my zip code is 78260. Can you find my order with that?

In [ ]:
import retail

retail.find_user_id_by_name_zip(first_name='Sofia', last_name='Li', zip_code='78260')

'sofia_li_9219'

In [ ]:
retail.get_user_details(user_id='sofia_li_9219')

{'name': {'first_name': 'Sofia', 'last_name': 'Li'},
 'address': {'address1': '786 Elm Street',
  'address2': 'Suite 546',
  'city': 'San Antonio',
  'country': 'USA',
  'state': 'TX',
  'zip': '78260'},
 'email': 'sofia.li7352@example.com',
 'payment_methods': {'paypal_8194385': {'id': 'paypal_8194385',
   'source': 'paypal'},
  'credit_card_3951670': {'id': 'credit_card_3951670',
   'source': 'credit_card',
   'brand': 'visa',
   'last_four': '6791'},
  'credit_card_8105988': {'id': 'credit_card_8105988',
   'source': 'credit_card',
   'brand': 'mastercard',
   'last_four': '8484'}},
 'orders': ['#W4689314', '#W8855135', '#W3916020', '#W5416052']}

In [ ]:
retail.get_order_details(order_id='#W3916020')

{'order_id': '#W3916020',
 'user_id': 'sofia_li_9219',
 'address': {'address1': '285 Elm Street',
  'address2': 'Suite 121',
  'city': 'Fort Worth',
  'country': 'USA',
  'state': 'TX',
  'zip': '76155'},
 'items': [{'name': 'Jigsaw Puzzle',
   'product_id': '1808611083',
   'item_id': '4068787148',
   'price': 52.01,
   'options': {'pieces': '500',
    'theme': 'art',
    'difficulty level': 'intermediate'}},
  {'name': 'Bicycle',
   'product_id': '9783735446',
   'item_id': '7758198585',
   'price': 1917.21,
   'options': {'frame size': 'medium', 'color': 'green', 'type': 'road'}}],
 'fulfillments': [{'tracking_id': ['580826001577'],
   'item_ids': ['4068787148', '7758198585']}],
 'status': 'delivered',
 'payment_history': [{'transaction_type': 'payment',
   'amount': 1969.22,
   'payment_method_id': 'credit_card_8105988'}]}

In [ ]:
retail.get_product_details(product_id='1808611083')

{'name': 'Jigsaw Puzzle',
 'product_id': '1808611083',
 'variants': {'3614853563': {'item_id': '3614853563',
   'options': {'pieces': '2000',
    'theme': 'art',
    'difficulty level': 'intermediate'},
   'available': False,
   'price': 46.99},
  '4772738468': {'item_id': '4772738468',
   'options': {'pieces': '1000',
    'theme': 'animals',
    'difficulty level': 'beginner'},
   'available': False,
   'price': 53.91},
  '4068787148': {'item_id': '4068787148',
   'options': {'pieces': '500',
    'theme': 'art',
    'difficulty level': 'intermediate'},
   'available': True,
   'price': 52.01},
  '3112842858': {'item_id': '3112842858',
   'options': {'pieces': '1000',
    'theme': 'fantasy',
    'difficulty level': 'intermediate'},
   'available': True,
   'price': 49.1},
  '7869640094': {'item_id': '7869640094',
   'options': {'pieces': '2000',
    'theme': 'animals',
    'difficulty level': 'expert'},
   'available': False,
   'price': 47.59},
  '1096508426': {'item_id': '1096508426'

In [ ]:
retail.get_product_details(product_id='9783735446')

{'name': 'Bicycle',
 'product_id': '9783735446',
 'variants': {'7758198585': {'item_id': '7758198585',
   'options': {'frame size': 'medium', 'color': 'green', 'type': 'road'},
   'available': True,
   'price': 1917.21},
  '5606522780': {'item_id': '5606522780',
   'options': {'frame size': 'large', 'color': 'red', 'type': 'mountain'},
   'available': True,
   'price': 1902.67},
  '6170152315': {'item_id': '6170152315',
   'options': {'frame size': 'small', 'color': 'red', 'type': 'mountain'},
   'available': False,
   'price': 1814.72},
  '3624655057': {'item_id': '3624655057',
   'options': {'frame size': 'medium', 'color': 'blue', 'type': 'road'},
   'available': True,
   'price': 2195.04},
  '2143041831': {'item_id': '2143041831',
   'options': {'frame size': 'medium', 'color': 'black', 'type': 'mountain'},
   'available': True,
   'price': 2076.5}}}

**Action Agent**: I can help you with your exchange. What would you like to exchange the jigsaw puzzle for? The available options are for different piece counts, themes, and difficulty levels. What about the bicycle? The available options are for different frame sizes, colors, and types.

**Simulated User**: For the bicycle, I'd like a larger frame size. For the puzzle, I want one with 1000 more pieces and the same difficulty. An animal theme would be great if you have one.

**Action Agent**: I have found the following items for your exchange:
- A 1500-piece, animal-themed, intermediate-difficulty jigsaw puzzle.
- A large-frame, red, mountain-type bicycle.

The price difference for this exchange will be a refund of \$20.55. Do you want to proceed with this exchange? If so, please confirm the payment method for the refund.

**Simulated User**: Yes, please proceed with the original payment method.

In [ ]:
retail.exchange_delivered_order_items(order_id='#W3916020', item_ids=['4068787148', '7758198585'], new_item_ids=['6245746168', '5606522780'], payment_method_id='credit_card_8105988')

{'order_id': '#W3916020',
 'user_id': 'sofia_li_9219',
 'address': {'address1': '285 Elm Street',
  'address2': 'Suite 121',
  'city': 'Fort Worth',
  'country': 'USA',
  'state': 'TX',
  'zip': '76155'},
 'items': [{'name': 'Jigsaw Puzzle',
   'product_id': '1808611083',
   'item_id': '4068787148',
   'price': 52.01,
   'options': {'pieces': '500',
    'theme': 'art',
    'difficulty level': 'intermediate'}},
  {'name': 'Bicycle',
   'product_id': '9783735446',
   'item_id': '7758198585',
   'price': 1917.21,
   'options': {'frame size': 'medium', 'color': 'green', 'type': 'road'}}],
 'fulfillments': [{'tracking_id': ['580826001577'],
   'item_ids': ['4068787148', '7758198585']}],
 'status': 'exchange requested',
 'payment_history': [{'transaction_type': 'payment',
   'amount': 1969.22,
   'payment_method_id': 'credit_card_8105988'}],
 'exchange_items': ['4068787148', '7758198585'],
 'exchange_new_items': ['5606522780', '6245746168'],
 'exchange_payment_method_id': 'credit_card_810598

In [ ]:
retail.get_order_details(order_id='#W4689314')

{'order_id': '#W4689314',
 'user_id': 'sofia_li_9219',
 'address': {'address1': '786 Elm Street',
  'address2': 'Suite 546',
  'city': 'San Antonio',
  'country': 'USA',
  'state': 'TX',
  'zip': '78260'},
 'items': [{'name': 'Digital Camera',
   'product_id': '8940227892',
   'item_id': '5996159312',
   'price': 2895.55,
   'options': {'resolution': '24MP', 'zoom': '3x', 'storage': 'SD card'}}],
 'fulfillments': [{'tracking_id': ['689944590938'],
   'item_ids': ['5996159312']}],
 'status': 'delivered',
 'payment_history': [{'transaction_type': 'payment',
   'amount': 2895.55,
   'payment_method_id': 'credit_card_8105988'}]}

In [ ]:
retail.get_product_details(product_id='8940227892')

{'name': 'Digital Camera',
 'product_id': '8940227892',
 'variants': {'6384525445': {'item_id': '6384525445',
   'options': {'resolution': '30MP', 'zoom': '5x', 'storage': 'CF card'},
   'available': True,
   'price': 2929.62},
  '3892645120': {'item_id': '3892645120',
   'options': {'resolution': '30MP', 'zoom': '10x', 'storage': 'CF card'},
   'available': False,
   'price': 3070.64},
  '1804581713': {'item_id': '1804581713',
   'options': {'resolution': '30MP', 'zoom': '3x', 'storage': 'SD card'},
   'available': True,
   'price': 2875.61},
  '9644439410': {'item_id': '9644439410',
   'options': {'resolution': '20MP', 'zoom': '5x', 'storage': 'CF card'},
   'available': True,
   'price': 3280.31},
  '5996159312': {'item_id': '5996159312',
   'options': {'resolution': '24MP', 'zoom': '3x', 'storage': 'SD card'},
   'available': True,
   'price': 2895.55},
  '7255224608': {'item_id': '7255224608',
   'options': {'resolution': '30MP', 'zoom': '3x', 'storage': 'CF card'},
   'available'

**Action Agent**: I can help you exchange the camera in order #W4689314. The current camera has a 24MP resolution, 3x zoom, and SD card storage. What would you like to exchange it for? There are other options with different resolutions, zoom levels, and storage types.

**Simulated User**: I'd like to exchange it for one with a slightly lower resolution.

**Action Agent**: I have found two cameras with a lower resolution (20MP):

1. A camera with 5x zoom and CF card storage for \$3280.31. This would cost an additional \$384.76.
2. A camera with 3x zoom and SD card storage for \$2823.96. You would receive a refund of \$71.59.

Which of these would you like to exchange for?

**Simulated User**: I'll take the second one, the one with the 3x zoom and SD card storage.

**Action Agent**: You've chosen to exchange your camera for the 20MP model with 3x zoom and SD card storage. You will receive a refund of \$71.59. Do you want the refund to be issued to your Mastercard ending in 8484, or your Visa ending in 6791?

**Simulated User**: Please use the other card this time.

In [ ]:
retail.get_user_details(user_id="sofia_li_9219")

{'name': {'first_name': 'Sofia', 'last_name': 'Li'},
 'address': {'address1': '786 Elm Street',
  'address2': 'Suite 546',
  'city': 'San Antonio',
  'country': 'USA',
  'state': 'TX',
  'zip': '78260'},
 'email': 'sofia.li7352@example.com',
 'payment_methods': {'paypal_8194385': {'id': 'paypal_8194385',
   'source': 'paypal'},
  'credit_card_3951670': {'id': 'credit_card_3951670',
   'source': 'credit_card',
   'brand': 'visa',
   'last_four': '6791'},
  'credit_card_8105988': {'id': 'credit_card_8105988',
   'source': 'credit_card',
   'brand': 'mastercard',
   'last_four': '8484'}},
 'orders': ['#W4689314', '#W8855135', '#W3916020', '#W5416052']}

In [ ]:
retail.exchange_delivered_order_items(order_id='#W4689314', item_ids=['5996159312'], new_item_ids=['8363011723'], payment_method_id='credit_card_3951670')

{'order_id': '#W4689314',
 'user_id': 'sofia_li_9219',
 'address': {'address1': '786 Elm Street',
  'address2': 'Suite 546',
  'city': 'San Antonio',
  'country': 'USA',
  'state': 'TX',
  'zip': '78260'},
 'items': [{'name': 'Digital Camera',
   'product_id': '8940227892',
   'item_id': '5996159312',
   'price': 2895.55,
   'options': {'resolution': '24MP', 'zoom': '3x', 'storage': 'SD card'}}],
 'fulfillments': [{'tracking_id': ['689944590938'],
   'item_ids': ['5996159312']}],
 'status': 'exchange requested',
 'payment_history': [{'transaction_type': 'payment',
   'amount': 2895.55,
   'payment_method_id': 'credit_card_8105988'}],
 'exchange_items': ['5996159312'],
 'exchange_new_items': ['8363011723'],
 'exchange_payment_method_id': 'credit_card_3951670',
 'exchange_price_difference': -71.59}

In [ ]:
retail.get_order_details(order_id='#W8855135')

{'order_id': '#W8855135',
 'user_id': 'sofia_li_9219',
 'address': {'address1': '786 Elm Street',
  'address2': 'Suite 546',
  'city': 'San Antonio',
  'country': 'USA',
  'state': 'TX',
  'zip': '78260'},
 'items': [{'name': 'Air Purifier',
   'product_id': '3821016478',
   'item_id': '4035304400',
   'price': 504.19,
   'options': {'room size': 'medium',
    'filter type': 'HEPA',
    'features': 'smart sensors'}},
  {'name': 'Hiking Boots',
   'product_id': '7363354090',
   'item_id': '1437889264',
   'price': 258.09,
   'options': {'size': '7', 'material': 'synthetic', 'waterproof': 'no'}},
  {'name': 'Skateboard',
   'product_id': '1968349452',
   'item_id': '3098764622',
   'price': 202.13,
   'options': {'deck material': 'plastic',
    'length': '34 inch',
    'design': 'plain'}},
  {'name': 'Yoga Mat',
   'product_id': '4635925001',
   'item_id': '2733768059',
   'price': 94.38,
   'options': {'thickness': '6mm',
    'material': 'natural rubber',
    'color': 'pink'}}],
 'fulfi

**Action Agent**: I see you want to cancel the skateboard from order #W8855135. Unfortunately, I can only cancel the entire order, not individual items from it.

Would you like me to cancel the whole order? You can then place a new order for the items you still want. Alternatively, I can transfer you to a human agent who might be able to assist you with this.

**Simulated User**: Go ahead and cancel the entire order.

In [ ]:
retail.cancel_pending_order(order_id='#W8855135', reason='no longer needed')

{'order_id': '#W8855135',
 'user_id': 'sofia_li_9219',
 'address': {'address1': '786 Elm Street',
  'address2': 'Suite 546',
  'city': 'San Antonio',
  'country': 'USA',
  'state': 'TX',
  'zip': '78260'},
 'items': [{'name': 'Air Purifier',
   'product_id': '3821016478',
   'item_id': '4035304400',
   'price': 504.19,
   'options': {'room size': 'medium',
    'filter type': 'HEPA',
    'features': 'smart sensors'}},
  {'name': 'Hiking Boots',
   'product_id': '7363354090',
   'item_id': '1437889264',
   'price': 258.09,
   'options': {'size': '7', 'material': 'synthetic', 'waterproof': 'no'}},
  {'name': 'Skateboard',
   'product_id': '1968349452',
   'item_id': '3098764622',
   'price': 202.13,
   'options': {'deck material': 'plastic',
    'length': '34 inch',
    'design': 'plain'}},
  {'name': 'Yoga Mat',
   'product_id': '4635925001',
   'item_id': '2733768059',
   'price': 94.38,
   'options': {'thickness': '6mm',
    'material': 'natural rubber',
    'color': 'pink'}}],
 'fulfi

**Action Agent**: I have now completed all of your requests:

- In order #W3916020, I have exchanged the bicycle and jigsaw puzzle.
- In order #W4689314, I have exchanged the camera.
- I have cancelled order #W8855135.

Is there anything else I can help you with today?

# Final Assertion

1. Order #W3916020 has the status `exchange requested`.
2. Order #W4689314 has the status `exchange requested`.
3. The status of order #W8855135 is `cancelled`.

In [ ]:
import retail
from Scripts.assertions_utils import compare_strings  # keep using your existing tolerant string comparator

# =========================
# Utils (can be moved to Scripts/fa_utils.py)
# =========================

def _safe_get_order(order_id: str):
    """Fetch order details and return (details, api_error_str)."""
    try:
        details = retail.get_order_details(order_id=order_id)
        return details or {}, None
    except Exception as e:
        return {}, str(e)

def _status(details: dict) -> str:
    """Return order status as a safe, comparable string."""
    return (details or {}).get("status", "") or ""

def _normalize_items_list(x):
    """Ensure we always return a list."""
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]

def _extract_item_ids_from_collection(items) -> list:
    """
    Accepts:
      - a list of dicts like [{'id':'123', 'status':'exchanged'}, ...]
      - a list of ids like ['123', '456']
    Returns:
      - list of string ids
    """
    out = []
    for it in _normalize_items_list(items):
        if isinstance(it, dict):
            # typical keys to try
            cand = it.get("id") or it.get("item_id") or it.get("sku") or it.get("productId")
            if cand is not None:
                out.append(str(cand))
        else:
            out.append(str(it))
    return out

def _extract_expected_kind_item_ids(details: dict, kind: str) -> list:
    """
    Try several common shapes to retrieve exchanged/cancelled item ids.
    Supported shapes:
      details['exchanged_items'] / ['cancelled_items'] as list[str|dict]
      details['exchange']['items'] / details['cancellation']['items']
      details['items'] with per-item 'status' == 'exchanged'/'cancelled'
    """
    d = details or {}

    # 1) direct keys
    direct_key_candidates = [
        f"{kind}_items",              # 'exchanged_items' / 'cancelled_items'
        f"{kind}Items",               # camelCase
        kind,                         # sometimes APIs use 'exchange'/'cancelled' directly as list
    ]
    for k in direct_key_candidates:
        if k in d:
            return _extract_item_ids_from_collection(d[k])

    # 2) nested sections
    nested_key_candidates = [
        f"{kind}_summary",
        f"{kind}_detail",
        f"{kind}_details",
        f"{kind}Summary",
        f"{kind}Detail",
        f"{kind}Details",
        "exchange", "cancellation", "cancel", "exchanges", "cancellations"
    ]
    for k in nested_key_candidates:
        section = d.get(k)
        if isinstance(section, dict) and "items" in section:
            return _extract_item_ids_from_collection(section["items"])

    # 3) fallback: scan 'items' array by per-item status
    items = d.get("items") or d.get("line_items") or d.get("order_items")
    ids = []
    if isinstance(items, list):
        for it in items:
            if isinstance(it, dict):
                st = str(it.get("status", "")).lower()
                if kind == "exchanged" and ("exchang" in st or st == "exchanged"):
                    ids.append(str(it.get("id") or it.get("item_id") or it.get("sku")))
                if kind == "cancelled" and ("cancel" in st or st == "cancelled"):
                    ids.append(str(it.get("id") or it.get("item_id") or it.get("sku")))
    return [i for i in ids if i and i != "None"]

def _assert_status(order_id: str, expected_status: str, actual_status: str, api_error: str | None):
    ok = compare_strings(actual_status, expected_status)
    assert ok, (
        f"Order {order_id}: Status mismatch. "
        f"Expected '{expected_status}', got '{actual_status}'."
        + (f" API error: {api_error}" if api_error else "")
    )

def _assert_items(order_id: str, kind: str, expected_ids: list[str], actual_ids: list[str], api_error: str | None):
    # Compare as sets because order typically doesn't matter
    exp_set, act_set = set(map(str, expected_ids)), set(map(str, actual_ids))
    ok = exp_set == act_set
    assert ok, (
        f"Order {order_id}: {kind.capitalize()} item IDs mismatch. "
        f"Expected {sorted(exp_set)}, got {sorted(act_set)}."
        + (f" API error: {api_error}" if api_error else "")
    )

# =========================
# Test Data
# =========================

ORDER_ID_1 = "#W3916020"
EXPECTED_EXCHANGED_ITEM_IDS_1 = ['5606522780', '6245746168']

ORDER_ID_2 = "#W4689314"
EXPECTED_EXCHANGED_ITEM_IDS_2 = ['8363011723']  # keep as list for uniformity

ORDER_ID_3 = "#W8855135"
EXPECTED_STATUS_1 = 'exchange requested'
EXPECTED_STATUS_2 = 'exchange requested'
EXPECTED_STATUS_3 = 'cancelled'

# =========================
# Assertions
# =========================

# --- Order 1: status + exchanged items ---
order_1_details, api_err_1 = _safe_get_order(ORDER_ID_1)
_actual_status_1 = _status(order_1_details)
_assert_status(ORDER_ID_1, EXPECTED_STATUS_1, _actual_status_1, api_err_1)

actual_exchanged_ids_1 = _extract_expected_kind_item_ids(order_1_details, "exchanged")
_assert_items(ORDER_ID_1, "exchanged", EXPECTED_EXCHANGED_ITEM_IDS_1, actual_exchanged_ids_1, api_err_1)

# --- Order 2: status + exchanged items ---
order_2_details, api_err_2 = _safe_get_order(ORDER_ID_2)
_actual_status_2 = _status(order_2_details)
_assert_status(ORDER_ID_2, EXPECTED_STATUS_2, _actual_status_2, api_err_2)

actual_exchanged_ids_2 = _extract_expected_kind_item_ids(order_2_details, "exchanged")
_assert_items(ORDER_ID_2, "exchanged", EXPECTED_EXCHANGED_ITEM_IDS_2, actual_exchanged_ids_2, api_err_2)

# --- Order 3: status only (add expected cancelled items here if they’re known) ---
order_3_details, api_err_3 = _safe_get_order(ORDER_ID_3)
_actual_status_3 = _status(order_3_details)
_assert_status(ORDER_ID_3, EXPECTED_STATUS_3, _actual_status_3, api_err_3)

# If you later know exactly which items were cancelled for Order 3, uncomment:
# EXPECTED_CANCELLED_ITEM_IDS_3 = ['<id1>', '<id2>']
# actual_cancelled_ids_3 = _extract_expected_kind_item_ids(order_3_details, "cancelled")
# _assert_items(ORDER_ID_3, "cancelled", EXPECTED_CANCELLED_ITEM_IDS_3, actual_cancelled_ids_3, api_err_3)
